importing Required modules

In [172]:
import pandas as pd
import numpy as np
import sklearn
from sklearn import preprocessing as per
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt 
from sklearn_pandas import DataFrameMapper
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.decomposition import TruncatedSVD

from lifelines.utils import concordance_index
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,median_absolute_error


import deepsurvk
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNetCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [173]:
#READING CSV FILE
df1 = pd.read_csv("data_bcr_clinical_data_patient.csv",na_values='?')
#EXCEPT CLINICAL DATA OTHERS HAVE PATIENT IDs WITH -01, SO ADD -01 AT THE END
df1.at[4,"Patient Identifier"]
def ankfunc(s):
    return s+"-01"
for i in range(4,532):
    df1.at[i,"Patient Identifier"]=ankfunc(df1.at[i,"Patient Identifier"])

#DROP ROWS AND COLUMNS
df1.drop([0,1,2,3] , inplace=True)
df1.set_index("Patient Identifier", inplace=True)

    
df1.replace('unknown',np.nan , inplace=True)
df1.replace('[Not Available]',np.nan , inplace=True)

df1.fillna(df1.mean(), inplace=True)
df1

C:\Users\PRODEE~1\AppData\Local\Temp/ipykernel_2452/2649448865.py:18: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df1.fillna(df1.mean(), inplace=True)


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NaN,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NaN,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [174]:
df1.fillna(method='ffill', inplace=True)
df1.fillna(method='bfill', inplace=True)
df1

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,1:Recurred/Progressed,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NOT HISPANIC OR LATINO,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NOT HISPANIC OR LATINO,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [175]:
df1['Lymph node neck dissection indicator'].replace(['[Not Available]','NO','YES'],['00','1','2'],inplace=True)

df1['Overall Survival Status'].replace(['0:LIVING','1:DECEASED'],['0','1'],inplace=True)
df1['Patient Primary Tumor Site'].replace(['[Not Available]','Buccal Mucosa','Larynx','Oral Cavity','Floor of mouth','Tonsil','Hypopharynx','Alveolar Ridge','Hard Palate','Oropharynx','Lip','Base of tongue','Oral Tongue'],['00','1','2','3','4','5','6','7','8','9','10','11','12'],inplace=True)
df1['Sex'].replace(['[Not Available]','Male','Female'],['00','1','2'],inplace=True)
df1['Race Category'].replace(['[Not Available]','WHITE','BLACK OR AFRICAN AMERICAN','ASIAN','AMERICAN INDIAN OR ALASKA NATIVE'],['00','1','2','3','4'],inplace=True)
df1['Ethnicity Category'].replace(['[Not Available]','NOT HISPANIC OR LATINO','HISPANIC OR LATINO'],['00','1','2'],inplace=True)
df1['Prior Cancer Diagnosis Occurence'].replace(['[Not Available]','No','Yes','Yes, History of Synchronous/Bilateral Malignancy','Yes, History of Prior Malignancy'],['00','1','2','3','4'],inplace=True)
df1['Neoadjuvant Therapy Type Administered Prior To Resection Text'].replace(['[Not Available]','No','Yes'],['00','1','2'],inplace=True)
df1['Vital Status'].replace(['[Not Available]','Dead','Alive'],['00','1','2'],inplace=True)
df1['American Joint Committee on Cancer Publication Version Type'].replace(['[Not Available]','6th','7th','5th','4th'],['00','1','2','3','4'],inplace=True)
df1['American Joint Committee on Cancer Tumor Stage Code'].replace(['[Not Available]','T0','T1','T2','T3','T4','T4a','T4b','TX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Disease Free Status'].replace(['0:DiseaseFree','1:Recurred/Progressed','[Not Available]'],['0','1','00'],inplace=True)
df1['Neoplasm Histologic Grade'].replace(['[Not Available]','G1','G2','G3','GX','G4'],['00','1','2','3','4','5'],inplace=True)
df1['Alcohol History Documented'].replace(['[Not Available]','No','Yes','NO','YES'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','N0','N1','N2','N2a','N2b','N2c','N3','NX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm Disease Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','Discrepancy','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','00','1','2','3','4','5','6'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage'].replace(['[Not Available]','M1','M1','MX','M0'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage'].replace(['[Not Available]','N0','N1','N2a','N2b','N2c','N3','NX','N2'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage'].replace(['[Not Available]','T1','T2','T3','T4a','T4b','TX','T4'],['00','1','2','3','4','5','6','7'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Group Stage'].replace(['[Not Available]','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','1','2','3','4','5','6'],inplace=True)

df1.head()

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,12,1,2,1,1,1,2013,2,2,2,...,66,4,3,4,4,0,0,3.35,0,3.35
TCGA-BA-4074-01,12,1,1,1,1,1,2003,2,1,1,...,69,4,5,3,4,0,1,15.18,1,13.01
TCGA-BA-4075-01,12,1,2,1,2,2,2004,2,1,1,...,49,4,2,4,4,0,1,9.3,1,7.75
TCGA-BA-4076-01,2,1,1,1,1,1,2003,2,1,1,...,39,4,5,3,4,0,1,13.63,1,9.4
TCGA-BA-4077-01,11,2,1,1,2,2,2003,2,1,1,...,45,4,6,5,5,0,1,37.25,1,9.4


In [176]:
#STORING REDUCED DATA TO CSV
cl = pd.DataFrame(df1)
cl.to_csv("CLINICALpreprocessed.csv")
print("Data exported to csv file")

Data exported to csv file


In [177]:
#READING CSV FILE
df =pd.read_csv("data_methylation_hm450.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df2 = df.T
df2.to_csv("Methylation.csv")
df2.head()

Hugo_Symbol,TSEN34,MUSTN1,C3orf16,CKLF,SFRS7,FAM180B,PTPRF,C6orf168,LOC728024,DSTYK,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
TCGA-4P-AA8J-01,0.160658,0.856690,0.689981,0.063268,0.088333,0.738956,0.689032,0.485472,0.847822,0.018144,...,0.021239,0.095778,0.061835,0.043785,0.055341,0.066497,0.080675,0.041548,0.054317,0.098915
TCGA-BA-4074-01,0.172720,0.888797,0.448310,0.095680,0.054274,0.506644,0.842746,0.188788,0.916802,0.018413,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
TCGA-BA-4075-01,0.091838,0.876359,0.336352,0.079018,0.062922,0.475571,0.783786,0.221566,0.792347,0.021204,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
TCGA-BA-4076-01,0.127324,0.911893,0.757925,0.095460,0.073372,0.834641,0.718938,0.791580,0.898727,0.014496,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
TCGA-BA-4077-01,0.132946,0.893790,0.556940,0.074819,0.080927,0.773512,0.392731,0.283078,0.857577,0.018026,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227


In [178]:
# Merge datasets based on the patient identifier
merged_df = pd.merge(cl, df2, left_index=True, right_index=True, how="inner")
df2=merged_df


In [179]:

c2 = pd.DataFrame(df2)
c2.to_csv("Methylationpreprocessed.csv")

In [180]:

# Extract target variable (survival time) from clinical data
y = c2['Overall Survival (Months)']
c2 = c2.drop('Overall Survival (Months)', axis=1)


In [181]:
y.drop(y.index[-1], inplace=True)
dm=c2.iloc[:,:]
#print(d)
dm = dm.reset_index()
M=dm.iloc[1:,1:]
M


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
1,12,1,1,1,1,1,2003,2,1,1,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
2,12,1,2,1,2,2,2004,2,1,1,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
3,2,1,1,1,1,1,2003,2,1,1,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
4,11,2,1,1,2,2,2003,2,1,1,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227
5,2,1,1,1,1,1,2003,2,1,1,...,0.036186,0.065871,0.042553,0.043962,0.036412,0.131708,0.062926,0.043616,0.044998,0.057528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,4,2,1,1,1,1,2009,2,1,1,...,0.019317,0.045418,0.032703,0.032664,0.035815,0.044621,0.080887,0.037113,0.033082,0.048041
524,6,2,1,1,3,1,2011,2,1,2,...,0.030936,0.068638,0.042534,0.043967,0.047700,0.063594,0.094378,0.045045,0.049718,0.084455
525,12,1,1,2,1,1,2013,2,2,2,...,0.024656,0.044608,0.030440,0.047566,0.064576,0.062233,0.051034,0.034986,0.053662,0.051514
526,4,1,1,1,1,1,2012,2,1,2,...,0.026150,0.078658,0.047960,0.048287,0.050811,0.059785,0.077223,0.051496,0.048270,0.059510


In [182]:
#standardize the data
scaler = StandardScaler()
X1 = scaler.fit_transform(M[:])
#PCA 
# fit pca on data
pca = TruncatedSVD(n_components=387, algorithm='randomized')


Z1=pca.fit_transform(X1)



In [183]:
n1 = pd.DataFrame(Z1)
print(n1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
n1["Hugo"]=hugo
n1

(527, 387)


,0,1,2,3,4,5,6,7,8,9,...,378,379,380,381,382,383,384,385,386,Hugo
0,37.966740,27.460375,-1.044253,-32.311313,-17.216051,-35.772245,15.875630,-5.593684,-7.900892,-8.992877,...,0.728494,0.727242,0.424744,-0.119890,1.759012,0.204576,-0.518604,1.069520,0.436908,TCGA-BA-4074-01
1,-11.652303,86.681811,47.471653,-7.123739,28.939181,-15.349657,-11.138881,-10.460588,6.572495,-2.113885,...,-0.089692,-0.018101,0.047921,0.184026,-0.070491,-0.163368,-0.122178,-0.075924,0.456240,TCGA-BA-4075-01
2,-52.592312,-15.064044,61.067690,-4.594665,25.017990,-14.316248,-21.350068,11.244861,0.231650,2.776049,...,0.765202,0.923339,-0.830486,0.281280,-0.253682,-0.272209,2.185746,0.099795,-0.238075,TCGA-BA-4076-01
3,5.490346,-8.517104,28.211523,26.724543,4.364628,-3.053363,-14.871908,-32.010688,3.442884,13.074267,...,-2.023631,1.340337,-3.184167,2.303492,0.787770,3.065275,-0.077991,2.699167,1.098164,TCGA-BA-4077-01
4,24.812767,-30.783921,45.110594,57.120804,-13.997583,-22.305011,-13.343090,13.943742,3.410954,5.255301,...,0.238238,0.000091,0.868968,-0.927129,-0.695315,-0.135199,-0.011282,-0.620350,0.131763,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-19.000191,-0.087101,-7.152378,-28.030376,12.615967,52.431953,0.527690,20.290588,8.299339,-3.487190,...,-5.156459,0.274117,2.062968,3.151928,0.733261,-0.627969,0.049742,0.238390,0.604417,TCGA-UF-A7JT-01
523,-26.270457,19.437747,-19.179308,-21.005025,16.054569,12.828012,9.676075,5.432176,-20.396517,-11.095898,...,1.289206,0.171627,-0.359852,2.382359,-0.990931,0.038931,2.166305,-1.033819,2.309722,TCGA-UF-A7JV-01
524,23.757266,-9.124223,39.910962,52.611628,-20.907197,63.207499,6.797034,-17.175075,-22.865997,-15.471139,...,-0.632070,0.769030,0.266473,-0.262037,-0.478271,0.810225,0.250285,-0.919875,0.717013,TCGA-UP-A6WW-01
525,77.957372,29.350011,-36.095846,22.874074,102.381682,87.737924,-33.493274,0.916156,48.986830,-28.932604,...,-0.162564,-0.133092,0.436241,0.154374,0.244252,-0.086014,0.443458,0.115984,0.154820,TCGA-WA-A7GZ-01


In [184]:

lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z1, y)
n_components = Z1.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(310).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zl1 = np.column_stack(columns)
nl1 = pd.DataFrame(Zl1)
print(nl1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nl1["Hugo"]=hugo
nl1

Number of selected features:  310
Selected features:  Int64Index([373, 241, 307, 327, 325, 235, 342, 236, 300, 159,
            ...
             17, 107, 221, 310, 371, 100, 348, 110,  26, 131],
           dtype='int64', length=310)
(527, 125)


,0,1,2,3,4,5,6,7,8,9,...,116,117,118,119,120,121,122,123,124,Hugo
0,1.247918,-3.695102,-2.605677,0.058980,-0.384990,6.939106,3.737509,3.230926,-6.231843,-1.421956,...,-1.777080,0.925619,3.617435,-3.158499,-5.194502,2.690969,-3.953362,-2.496347,0.872262,TCGA-BA-4074-01
1,-0.190715,-0.190505,-0.208940,0.029281,-0.143974,0.643636,0.029349,-0.457831,0.497028,-0.322772,...,-0.752834,-0.211610,0.131504,0.790097,-0.054379,0.266007,0.159469,-0.018831,0.307819,TCGA-BA-4075-01
2,-0.362786,-1.644846,1.430387,-0.182344,-0.049258,1.475226,-0.845414,0.065583,0.076524,-0.090835,...,0.453855,0.917503,6.501736,-15.400918,-0.415835,0.608407,-1.564810,-4.583053,-0.733237,TCGA-BA-4076-01
3,2.905674,-0.103299,0.585129,7.506755,-5.695076,3.308305,-4.700024,-4.905093,-1.965440,2.784226,...,-2.724096,-6.750960,-3.730643,-2.151759,-3.294528,-5.065752,2.959299,-1.286960,1.422307,TCGA-BA-4077-01
4,0.670753,0.277800,0.750905,0.315231,-1.036217,-0.970411,-0.131270,1.366584,1.024137,-13.005906,...,3.864540,3.126116,-0.469188,8.208508,-1.136184,-0.560930,-0.824070,-3.992182,0.818545,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-1.622545,2.153456,-4.852505,-10.311071,5.353433,-3.138070,-2.131721,-2.881817,-1.704517,-1.054202,...,-5.286275,0.434867,-5.257400,2.492037,1.739169,-5.715017,0.709757,-2.401833,1.732592,TCGA-UF-A7JT-01
523,-0.255280,5.315167,-3.297381,-0.345253,3.024925,7.761184,-2.469798,5.132324,0.887981,1.538228,...,-1.796951,5.831081,4.118113,-10.712189,-0.257531,4.294719,-0.328059,6.408624,5.701880,TCGA-UF-A7JV-01
524,-0.283951,1.706808,1.682895,-0.493443,-2.670857,-2.527380,-1.597675,2.601307,-0.461197,-2.968083,...,-3.598760,1.994135,-3.967224,19.163322,-0.274156,-3.747042,-0.392930,9.433265,2.100559,TCGA-UP-A6WW-01
525,-0.463340,1.017301,0.689032,-0.178095,-0.791631,1.572736,-0.757322,-1.057875,0.282372,-0.401460,...,4.357019,-0.469352,3.774387,-0.995153,-0.327438,-0.975885,0.300002,1.909007,0.163624,TCGA-WA-A7GZ-01


In [185]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z1, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z1.shape[1])])
selected_features = coef.abs().nlargest(310).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Ze1 = np.column_stack(columns)
ne1 = pd.DataFrame(Ze1)
ne1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
ne1["Hugo"]=hugo
ne1


G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:614: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 161.60431604997575, tolerance: 37.95947935254157
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:614: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 391.0056380194874, tolerance: 37.95947935254157
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:614: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 490.8892526390264, tolerance: 37.95947935254157
  model = cd_fast.enet_coordinate_descent_gram(
G:\ana\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:614: ConvergenceWarning: Objective did not converge. You might want to increas

Number of selected features:  310
Selected features:  [7, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211

,0,1,2,3,4,5,6,7,8,9,...,301,302,303,304,305,306,307,308,309,Hugo
0,-5.593684,27.460375,-1.044253,-32.311313,-17.216051,-35.772245,15.875630,-7.900892,-8.992877,16.529898,...,3.056499,2.690969,-0.117045,-0.834611,0.279287,-2.605677,-1.037598,1.428938,0.874088,TCGA-BA-4074-01
1,-10.460588,86.681811,47.471653,-7.123739,28.939181,-15.349657,-11.138881,6.572495,-2.113885,1.708659,...,-0.296399,0.266007,0.143888,-0.439384,0.290374,-0.208940,0.260587,-0.057727,-0.062011,TCGA-BA-4075-01
2,11.244861,-15.064044,61.067690,-4.594665,25.017990,-14.316248,-21.350068,0.231650,2.776049,0.931249,...,-1.815660,0.608407,-1.944530,-0.244515,0.402343,1.430387,-0.869542,1.093529,-0.773653,TCGA-BA-4076-01
3,-32.010688,-8.517104,28.211523,26.724543,4.364628,-3.053363,-14.871908,3.442884,13.074267,-8.700608,...,3.248126,-5.065752,-0.659237,4.855076,-1.123151,0.585129,5.091533,1.659432,-7.922770,TCGA-BA-4077-01
4,13.943742,-30.783921,45.110594,57.120804,-13.997583,-22.305011,-13.343090,3.410954,5.255301,7.977272,...,-0.822076,-0.560930,0.510285,-0.621555,-0.864734,0.750905,0.972777,-2.873910,1.393144,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,20.290588,-0.087101,-7.152378,-28.030376,12.615967,52.431953,0.527690,8.299339,-3.487190,-12.992510,...,-5.694899,-5.715017,7.600001,-2.030437,0.153956,-4.852505,2.543005,-9.270734,-6.607807,TCGA-UF-A7JT-01
523,5.432176,19.437747,-19.179308,-21.005025,16.054569,12.828012,9.676075,-20.396517,-11.095898,2.527732,...,9.454664,4.294719,-1.013056,-1.166883,1.737582,-3.297381,0.454622,-1.716805,2.929741,TCGA-UF-A7JV-01
524,-17.175075,-9.124223,39.910962,52.611628,-20.907197,63.207499,6.797034,-22.865997,-15.471139,34.165472,...,-1.027453,-3.747042,1.959827,1.551958,0.113380,1.682895,-1.209081,-0.099512,-0.741569,TCGA-UP-A6WW-01
525,0.916156,29.350011,-36.095846,22.874074,102.381682,87.737924,-33.493274,48.986830,-28.932604,62.057055,...,-0.273655,-0.975885,-1.119904,-0.736821,-0.195518,0.689032,1.154620,-0.144836,0.810354,TCGA-WA-A7GZ-01


In [186]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z1, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z1.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(310)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zrf1 = np.column_stack(columns)
nrf1 = pd.DataFrame(Zrf1)
nrf1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nrf1["Hugo"]=hugo
nrf1

Number of selected features:  310
Selected features:  [322, 161, 171, 217, 216, 112, 363, 248, 37, 11, 21, 268, 178, 187, 7, 372, 342, 380, 373, 236, 273, 300, 321, 34, 83, 211, 376, 364, 169, 13, 381, 3, 42, 272, 306, 377, 384, 281, 130, 195, 28, 151, 33, 8, 133, 32, 118, 38, 10, 91, 165, 22, 36, 290, 357, 311, 309, 168, 123, 177, 96, 332, 136, 19, 323, 16, 230, 257, 1, 110, 159, 202, 210, 358, 176, 256, 190, 359, 369, 145, 337, 374, 199, 345, 366, 200, 184, 41, 334, 158, 378, 330, 172, 146, 114, 288, 62, 92, 164, 73, 124, 35, 80, 31, 340, 243, 205, 292, 180, 352, 341, 154, 6, 375, 316, 119, 277, 175, 47, 49, 213, 162, 336, 207, 333, 278, 51, 301, 197, 370, 102, 135, 302, 108, 279, 350, 241, 191, 126, 132, 355, 203, 153, 291, 66, 225, 351, 196, 18, 84, 255, 264, 86, 20, 297, 249, 239, 220, 387, 379, 385, 56, 72, 198, 259, 55, 328, 250, 174, 206, 382, 50, 99, 45, 303, 43, 261, 258, 331, 227, 265, 2, 128, 245, 78, 103, 283, 101, 324, 69, 347, 142, 383, 140, 30, 367, 218, 319, 63, 131, 1

,0,1,2,3,4,5,6,7,8,9,...,149,150,151,152,153,154,155,156,157,Hugo
0,2.733847,-2.587752,1.822771,10.947284,12.894657,-5.300243,-0.574215,-5.589267,0.399749,8.126443,...,6.085530,-2.083904,0.716521,9.795057,-3.892284,-0.262480,2.540931,3.346067,-0.341278,TCGA-BA-4074-01
1,-0.154937,0.475887,0.858620,-0.701929,-0.644878,2.534251,0.099187,-0.239981,11.904057,-2.036876,...,-1.251419,-0.442635,0.199853,2.132126,22.993916,0.301830,-0.088933,0.379551,-0.220468,TCGA-BA-4075-01
2,-1.511453,-2.630903,-1.575001,0.842195,2.690095,-1.404234,0.382059,-0.945430,7.486296,-3.040855,...,-1.481913,0.697213,-0.824886,-21.630341,-15.279113,-0.281665,-3.160127,1.949612,-3.506119,TCGA-BA-4076-01
3,-3.217211,-0.783696,-2.170645,-1.184324,-4.121405,2.378175,1.855115,6.352744,-1.406199,-2.625206,...,-1.977934,4.493165,-1.913367,-1.844887,2.917430,-8.903674,-3.129141,-5.900927,-4.871621,TCGA-BA-4077-01
4,1.180647,7.652272,3.617465,1.531650,-6.895103,-6.408005,-0.676286,0.343059,8.467468,0.243871,...,2.707131,-4.463145,1.713714,-5.396001,-11.344405,1.629967,2.458604,-3.850251,-1.217806,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,1.903963,0.205038,-2.028052,1.494987,0.704559,1.676264,-5.172160,5.346609,-3.010307,2.995235,...,1.499970,-2.411125,-6.467818,-3.902036,2.398456,-0.855013,-2.115897,-4.939887,1.076283,TCGA-UF-A7JT-01
523,1.472321,3.822100,-2.805935,-9.752400,1.155450,-7.549925,-0.587636,4.018721,-5.438433,12.014618,...,5.590883,7.741665,7.665074,-6.109072,8.703846,5.844629,-7.536163,2.907760,-0.545783,TCGA-UF-A7JV-01
524,-0.841177,-1.601464,3.850273,-1.418469,-2.491611,-19.690794,-0.859277,-0.067174,-18.094845,-4.376453,...,-11.122138,1.550034,2.871932,-1.643823,-18.394549,-0.863712,3.120022,-0.439864,-1.964856,TCGA-UP-A6WW-01
525,-0.005428,-0.018242,0.551518,-0.367297,-1.865859,-0.963219,0.597087,0.007168,20.267575,-21.179768,...,-0.049863,0.514155,-0.240216,2.828176,13.782212,0.179362,-0.361790,1.194629,0.411105,TCGA-WA-A7GZ-01


In [187]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=310)
rfe.fit(Z1, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zre1 = np.column_stack(columns)
nre1 = pd.DataFrame(Zre1)
nre1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nre1["Hugo"]=hugo
nre1

Selected Features: [6, 7, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 24, 28, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 47, 49, 50, 51, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 67, 68, 69, 70, 72, 73, 74, 75, 77, 78, 79, 80, 82, 83, 84, 85, 86, 87, 89, 90, 91, 92, 94, 96, 98, 100, 101, 102, 103, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 117, 118, 119, 120, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 133, 135, 136, 137, 138, 139, 141, 142, 144, 145, 146, 148, 149, 150, 151, 152, 154, 155, 156, 158, 159, 160, 161, 162, 163, 165, 166, 168, 169, 171, 173, 174, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 199, 200, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 232, 233, 235, 236, 237, 238, 240, 241, 243, 246, 247, 249, 250, 251, 252, 253, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 268, 269, 270, 2

,0,1,2,3,4,5,6,7,8,9,...,300,301,302,303,304,305,306,307,308,Hugo
0,15.875630,-5.593684,16.529898,-5.903702,-2.127261,-0.377044,2.060891,-3.072572,7.159269,-10.640677,...,0.728494,0.727242,0.424744,-0.119890,1.759012,0.204576,-0.518604,1.069520,0.436908,TCGA-BA-4074-01
1,-11.138881,-10.460588,1.708659,-24.139966,-7.474749,-43.920955,-7.678469,-40.083581,43.041716,-26.332798,...,-0.089692,-0.018101,0.047921,0.184026,-0.070491,-0.163368,-0.122178,-0.075924,0.456240,TCGA-BA-4075-01
2,-21.350068,11.244861,0.931249,20.151574,-34.465105,9.910241,41.344078,-5.257676,-5.649413,4.613421,...,0.765202,0.923339,-0.830486,0.281280,-0.253682,-0.272209,2.185746,0.099795,-0.238075,TCGA-BA-4076-01
3,-14.871908,-32.010688,-8.700608,5.388567,-6.954599,-10.499643,22.467997,-12.405528,6.682474,-2.289595,...,-2.023631,1.340337,-3.184167,2.303492,0.787770,3.065275,-0.077991,2.699167,1.098164,TCGA-BA-4077-01
4,-13.343090,13.943742,7.977272,-17.489836,15.237576,-6.241609,-6.530951,-18.147118,3.272824,15.361730,...,0.238238,0.000091,0.868968,-0.927129,-0.695315,-0.135199,-0.011282,-0.620350,0.131763,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.527690,20.290588,-12.992510,4.323593,4.333409,-9.763697,-1.249040,-14.949191,-2.416083,2.062314,...,-5.156459,0.274117,2.062968,3.151928,0.733261,-0.627969,0.049742,0.238390,0.604417,TCGA-UF-A7JT-01
523,9.676075,5.432176,2.527732,8.894714,-4.306876,-27.764684,-9.741122,-1.571661,12.478862,4.555462,...,1.289206,0.171627,-0.359852,2.382359,-0.990931,0.038931,2.166305,-1.033819,2.309722,TCGA-UF-A7JV-01
524,6.797034,-17.175075,34.165472,13.107518,12.415976,-14.272440,-8.248232,-5.568044,5.124039,-2.816837,...,-0.632070,0.769030,0.266473,-0.262037,-0.478271,0.810225,0.250285,-0.919875,0.717013,TCGA-UP-A6WW-01
525,-33.493274,0.916156,62.057055,7.912945,-44.169307,12.078353,-57.427969,-0.899998,12.325944,-16.564468,...,-0.162564,-0.133092,0.436241,0.154374,0.244252,-0.086014,0.443458,0.115984,0.154820,TCGA-WA-A7GZ-01


In [188]:
#READING CSV FILE
df = pd.read_csv("data_RNA_Seq_v2_expression_median.csv")

#DROP ROWS AND COLUMNS
df=df.replace(0,np.nan)
df=df.dropna()
df=df.replace(np.nan,0)
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df4 = df.T
df4.to_csv("RNAb.csv")
df4.head()
merged_df = pd.merge(cl, df4, left_index=True, right_index=True, how="inner")
df4=merged_df

In [189]:
c4 = pd.DataFrame(df4)
c4.to_csv("RNApreprocessed.csv")
y = c4['Overall Survival (Months)']
c4 = c4.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
drn=c4.iloc[:,:]
#print(d)
drn = drn.reset_index()
Rn=drn.iloc[1:,1:]
Rn.head()


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,LOC154274,ZW10,ZWILCH,ZWINT,ZXDB,LOC100130182,ZYG11B,ZYX,FLJ10821,ZZZ3
1,12,1,1,1,1,1,2003,2,1,1,...,311.0030,283.6409,2132.4595,1193.1417,172.2656,380.3717,805.0624,2516.9279,258.5911,1088.3179
2,12,1,2,1,2,2,2004,2,1,1,...,225.1105,512.3945,761.0023,673.1877,172.0488,562.2404,487.7395,5930.0549,292.6437,980.3028
3,2,1,1,1,1,1,2003,2,1,1,...,157.9431,307.4905,480.0682,1032.6643,324.2818,1440.9025,722.5502,2674.5376,672.1763,998.5570
4,11,2,1,1,2,2,2003,2,1,1,...,137.6323,361.4052,1325.3128,1620.3080,210.7796,1423.0029,770.9336,8035.6112,763.2339,692.9740
5,2,1,1,1,1,1,2003,2,1,1,...,241.8520,414.1231,874.1257,1145.1112,372.9953,2634.2473,780.1345,3895.2406,1556.6477,1309.6223


Applying PCA dimensioality reduction technique

In [190]:
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X3 = scaler.fit_transform(Rn[:])
#fit pca on data
pca = TruncatedSVD(n_components=323, algorithm='randomized')
pca.fit(X3)
#transform pca
Z3 =pca.transform(X3)
Z3


array([[ 60.11809869,  26.14025133, -64.18048127, ...,  -0.53391831,
          0.22077797,  -0.53268219],
       [ 43.62700301, -12.39704133, -37.88746569, ...,   0.1719634 ,
          0.32342931,   0.16194583],
       [  1.01490858, -16.49333337, -25.13506405, ...,  -0.10475231,
         -1.75194517,   0.1909342 ],
       ...,
       [ 78.47038304,  55.97978897,  10.62424179, ...,   0.70146509,
          1.13120466,  -0.14838291],
       [  6.35037415,   5.82697398, -12.86072595, ...,   0.87253974,
         -2.52329651,  -2.51747324],
       [-11.89950089,  14.3759827 ,  15.11476505, ...,  -1.8191358 ,
         -3.36844269,  -0.60574998]])

Applying RFE Feature selection methods

In [191]:
n3 = pd.DataFrame(Z3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
n3["Hugo"]=hugo
n3

,0,1,2,3,4,5,6,7,8,9,...,314,315,316,317,318,319,320,321,322,Hugo
0,60.118099,26.140251,-64.180481,84.128182,135.132266,118.150329,53.186133,59.027978,-17.676262,-3.691583,...,-0.289892,0.126189,0.534958,-0.573278,-0.642359,-0.201631,-0.533918,0.220778,-0.532682,TCGA-BA-4074-01
1,43.627003,-12.397041,-37.887466,67.227723,89.696953,67.235101,24.330854,26.330579,-15.242053,-4.606208,...,0.035952,-1.376173,-1.852517,-0.242694,0.073793,-0.976532,0.171963,0.323429,0.161946,TCGA-BA-4075-01
2,1.014909,-16.493333,-25.135064,17.473437,11.606667,25.737596,-9.259798,15.763583,-13.589997,-14.311950,...,0.156736,-1.259347,1.364739,0.259466,0.507788,0.896567,-0.104752,-1.751945,0.190934,TCGA-BA-4076-01
3,-5.197779,1.672591,3.526023,-12.276289,14.811709,-6.695149,7.641358,12.263671,-38.653667,-3.170848,...,0.207516,-3.476009,-0.776863,-2.967671,1.211419,-2.692659,2.655172,0.597315,-1.673360,TCGA-BA-4077-01
4,-23.492684,78.722428,2.399360,22.597803,-5.793599,11.976908,-21.316500,1.220622,-7.008985,-13.086680,...,0.067383,-0.077204,-2.681875,0.001546,0.588643,-1.417955,-0.357044,1.392118,0.908680,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,51.719029,-13.759235,9.603475,-17.378117,36.199802,-6.625736,5.532348,-31.437382,1.696102,0.101085,...,2.259700,1.341538,3.709266,-1.465861,-2.807047,-3.088989,-0.556149,1.349638,-1.683305,TCGA-UF-A7JT-01
515,13.009808,-14.443833,24.877834,12.637027,33.381403,-4.012832,-11.003321,-14.842029,9.239174,-18.849358,...,0.297082,-0.529923,0.950912,-1.620382,-0.041730,1.568605,1.280483,-0.701435,-0.642562,TCGA-UF-A7JV-01
516,78.470383,55.979789,10.624242,-30.658030,7.747516,-0.823795,-19.008927,24.587056,-10.211918,-5.948767,...,2.111221,0.135640,2.180954,0.217608,-1.294366,1.533609,0.701465,1.131205,-0.148383,TCGA-UP-A6WW-01
517,6.350374,5.826974,-12.860726,18.283309,-11.623426,-22.091447,-22.782900,-2.080459,9.482621,6.766271,...,-3.479158,-0.480167,-1.312885,-1.006995,-0.807323,-1.372672,0.872540,-2.523297,-2.517473,TCGA-WA-A7GZ-01


In [192]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z3, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z3.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(258)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zrf3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf3 = pd.DataFrame(Zrf3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nrf3["Hugo"]=hugo
nrf3

Number of selected features:  258
Selected features:  [298, 2, 305, 275, 1, 309, 254, 22, 299, 158, 302, 141, 114, 91, 274, 195, 66, 168, 46, 99, 3, 273, 98, 84, 312, 117, 290, 318, 79, 307, 227, 229, 115, 124, 181, 259, 241, 165, 269, 13, 202, 256, 23, 88, 260, 81, 105, 296, 205, 193, 221, 251, 94, 321, 83, 179, 287, 267, 138, 145, 8, 281, 44, 293, 323, 276, 73, 70, 244, 284, 308, 190, 178, 48, 236, 212, 10, 52, 268, 257, 319, 90, 242, 197, 282, 64, 316, 300, 28, 30, 67, 313, 38, 182, 301, 169, 322, 35, 116, 264, 59, 49, 100, 102, 262, 226, 255, 237, 214, 12, 315, 213, 140, 60, 75, 154, 232, 258, 311, 155, 243, 317, 45, 295, 118, 320, 283, 230, 192, 110, 41, 294, 303, 4, 306, 74, 219, 249, 131, 289, 270, 291, 170, 292, 187, 37, 248, 245, 261, 39, 151, 176, 76, 69, 200, 112, 231, 57, 123, 9, 139, 58, 20, 285, 97, 272, 194, 208, 186, 106, 191, 42, 156, 198, 149, 144, 33, 108, 183, 218, 148, 47, 279, 265, 36, 72, 204, 247, 228, 104, 78, 250, 310, 26, 34, 201, 239, 122, 77, 222, 137, 54, 

,0,1,2,3,4,5,6,7,8,9,...,55,56,57,58,59,60,61,62,63,Hugo
0,-0.297690,-64.180481,-0.110434,0.460578,26.140251,-0.541334,0.526302,-4.840718,-0.016289,-1.560900,...,3.510499,-0.027482,-0.338771,-0.615581,-5.380226,-17.676262,1.790448,3.434562,-0.433396,TCGA-BA-4074-01
1,-1.131100,-37.887466,-1.026985,1.373774,-12.397041,0.652455,-1.928405,14.676029,0.306599,1.380746,...,-4.263959,1.843897,2.643621,3.792931,10.390657,-15.242053,-2.583776,0.439883,-0.222730,TCGA-BA-4075-01
2,1.437402,-25.135064,0.178184,-0.879880,-16.493333,-0.634988,-0.886499,-7.031177,-0.761871,7.380313,...,1.199736,0.999420,1.628362,-2.954519,-3.340328,-13.589997,4.358036,0.353406,-0.442255,TCGA-BA-4076-01
3,1.302873,3.526023,3.268334,2.043752,1.672591,3.400235,1.125827,-4.691108,5.460884,-2.691055,...,2.907516,2.364759,-1.326852,-0.760148,-6.755697,-38.653667,0.763933,1.618367,-0.952728,TCGA-BA-4077-01
4,-0.288414,2.399360,-1.957533,-2.783829,78.722428,0.618983,3.988676,-9.582857,1.897340,-1.956268,...,2.087961,0.435094,2.038384,2.766459,3.002375,-7.008985,-1.421561,4.708957,-1.910264,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,2.701124,9.603475,1.679675,-3.418426,-13.759235,0.006229,-1.529892,-4.713486,1.283457,2.801984,...,2.449268,3.096592,1.277902,-0.322811,-1.022423,1.696102,1.774195,6.270422,3.125836,TCGA-UF-A7JT-01
515,1.651496,24.877834,-2.032022,2.060005,-14.443833,-1.073708,1.524770,-4.324369,-0.026980,12.140193,...,0.636444,-2.490723,-0.773444,3.342766,-9.294073,9.239174,0.172934,-3.448245,-1.441785,TCGA-UF-A7JV-01
516,0.082253,10.624242,0.089402,-0.464340,55.979789,0.568417,-2.699653,-14.842493,-1.572292,-0.711202,...,-3.298100,-0.093606,2.219228,-1.753116,-11.445094,-10.211918,-3.062364,4.992517,-0.821527,TCGA-UP-A6WW-01
517,2.032055,-12.860726,3.165689,-7.479713,5.826974,0.501657,-0.965043,0.996652,0.909525,-5.627142,...,-7.292591,-0.992255,1.404597,0.072745,-3.252443,9.482621,-1.168233,5.153247,-2.187939,TCGA-WA-A7GZ-01


In [193]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=258)
rfe.fit(Z3, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zre3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre3 = pd.DataFrame(Zre3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nre3["Hugo"]=hugo
nre3

Selected Features: [1, 5, 9, 10, 14, 15, 18, 19, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 34, 35, 37, 38, 39, 41, 44, 45, 46, 47, 48, 49, 53, 54, 55, 58, 59, 60, 61, 63, 64, 65, 66, 69, 71, 72, 74, 75, 76, 78, 79, 80, 81, 82, 84, 85, 86, 87, 89, 90, 91, 92, 93, 94, 96, 98, 99, 100, 101, 103, 104, 105, 107, 108, 109, 110, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 123, 124, 125, 126, 127, 129, 130, 131, 132, 133, 134, 136, 137, 138, 139, 142, 143, 144, 145, 146, 147, 148, 150, 151, 152, 153, 154, 155, 156, 157, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 193, 194, 195, 196, 198, 199, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 217, 218, 221, 222, 223, 225, 227, 228, 229, 230, 231, 233, 234, 235, 236, 237, 238, 240, 242, 244, 245, 246, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 262, 263, 264, 265, 266, 267, 268, 269, 270

,0,1,2,3,4,5,6,7,8,9,...,248,249,250,251,252,253,254,255,256,Hugo
0,26.140251,118.150329,-3.691583,21.816013,-4.723802,-17.608646,16.352260,29.073651,-5.578299,-4.840718,...,-1.538907,-0.289892,0.534958,-0.573278,-0.642359,-0.201631,-0.533918,0.220778,-0.532682,TCGA-BA-4074-01
1,-12.397041,67.235101,-4.606208,11.582840,-37.141295,-10.054701,45.799704,7.321194,-16.664683,14.676029,...,1.651486,0.035952,-1.852517,-0.242694,0.073793,-0.976532,0.171963,0.323429,0.161946,TCGA-BA-4075-01
2,-16.493333,25.737596,-14.311950,5.457589,-5.934960,-16.142699,30.708370,3.693780,8.070135,-7.031177,...,1.092793,0.156736,1.364739,0.259466,0.507788,0.896567,-0.104752,-1.751945,0.190934,TCGA-BA-4076-01
3,1.672591,-6.695149,-3.170848,-4.432494,2.890126,-8.861937,22.973296,3.040380,0.208434,-4.691108,...,-1.049366,0.207516,-0.776863,-2.967671,1.211419,-2.692659,2.655172,0.597315,-1.673360,TCGA-BA-4077-01
4,78.722428,11.976908,-13.086680,16.454448,-3.561151,10.785781,-12.780905,18.913480,-2.475433,-9.582857,...,-0.061464,0.067383,-2.681875,0.001546,0.588643,-1.417955,-0.357044,1.392118,0.908680,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-13.759235,-6.625736,0.101085,9.640869,8.643019,6.362297,-3.570042,0.410766,0.277465,-4.713486,...,-0.927907,2.259700,3.709266,-1.465861,-2.807047,-3.088989,-0.556149,1.349638,-1.683305,TCGA-UF-A7JT-01
515,-14.443833,-4.012832,-18.849358,-27.275862,5.127871,12.110279,3.915131,-4.693864,-16.332269,-4.324369,...,0.966388,0.297082,0.950912,-1.620382,-0.041730,1.568605,1.280483,-0.701435,-0.642562,TCGA-UF-A7JV-01
516,55.979789,-0.823795,-5.948767,-28.453676,-26.703515,-16.962199,1.276625,5.037647,-5.058007,-14.842493,...,-0.843507,2.111221,2.180954,0.217608,-1.294366,1.533609,0.701465,1.131205,-0.148383,TCGA-UP-A6WW-01
517,5.826974,-22.091447,6.766271,12.002829,-17.249402,-2.628596,-5.108419,21.547091,4.463423,0.996652,...,0.726819,-3.479158,-1.312885,-1.006995,-0.807323,-1.372672,0.872540,-2.523297,-2.517473,TCGA-WA-A7GZ-01


In [194]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z3, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z3.shape[1])])
selected_features = coef.abs().nlargest(258).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Ze3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne3 = pd.DataFrame(Ze3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
ne3["Hugo"]=hugo
ne3


Number of selected features:  258
Selected features:  [30, 23, 1, 9, 5, 3, 7, 2, 4, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211

,0,1,2,3,4,5,6,7,8,9,...,249,250,251,252,253,254,255,256,257,Hugo
0,1.747469,20.123539,26.140251,-3.691583,118.150329,84.128182,59.027978,-64.180481,135.132266,53.186133,...,-0.028270,0.707471,-1.717746,0.382920,0.526302,-0.340278,2.281521,-1.715018,-1.098971,TCGA-BA-4074-01
1,7.521388,29.372762,-12.397041,-4.606208,67.235101,67.227723,26.330579,-37.887466,89.696953,24.330854,...,-1.246431,-3.193501,0.566694,1.662102,-1.928405,0.142703,-0.487211,1.300993,1.957796,TCGA-BA-4075-01
2,-10.452968,-1.958085,-16.493333,-14.311950,25.737596,17.473437,15.763583,-25.135064,11.606667,-9.259798,...,-0.511855,-0.117125,0.549376,-0.790516,-0.886499,-0.116986,-4.107995,2.271460,1.696570,TCGA-BA-4076-01
3,6.123146,-5.454031,1.672591,-3.170848,-6.695149,-12.276289,12.263671,3.526023,14.811709,7.641358,...,2.188717,0.289284,6.115356,-0.209474,1.125827,-4.161005,-1.130470,0.423584,0.152112,TCGA-BA-4077-01
4,1.632842,10.283374,78.722428,-13.086680,11.976908,22.597803,1.220622,2.399360,-5.793599,-21.316500,...,-0.524025,-5.668230,-1.582955,2.847079,3.988676,0.341831,2.877113,4.623115,-1.587577,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-1.499458,9.097573,-13.759235,0.101085,-6.625736,-17.378117,-31.437382,9.603475,36.199802,5.532348,...,-2.941705,-2.981876,-2.347586,-0.363085,-1.529892,2.331736,1.542222,1.624299,-3.883139,TCGA-UF-A7JT-01
515,-3.103691,-3.999083,-14.443833,-18.849358,-4.012832,12.637027,-14.842029,24.877834,33.381403,-11.003321,...,-1.505102,3.601498,-2.211584,4.856141,1.524770,3.650622,3.440829,-1.307714,0.797720,TCGA-UF-A7JV-01
516,2.633177,6.821034,55.979789,-5.948767,-0.823795,-30.658030,24.587056,10.624242,7.747516,-19.008927,...,-0.674003,-1.872907,-0.391577,4.111129,-2.699653,-5.387747,-0.951998,-1.504135,-5.004875,TCGA-UP-A6WW-01
517,-0.740219,15.854177,5.826974,6.766271,-22.091447,18.283309,-2.080459,-12.860726,-11.623426,-22.782900,...,2.314263,-2.094932,2.863428,-1.510014,-0.965043,-2.725356,1.383889,2.086629,1.753995,TCGA-WA-A7GZ-01


In [195]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z3, y)
n_components = Z3.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(258).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zl3 = np.column_stack(columns)
nl3 = pd.DataFrame(Zl3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nl3["Hugo"]=hugo
nl3

Number of selected features:  258
Selected features:  Int64Index([305, 293, 275, 299, 250, 309, 268, 274, 276, 282,
            ...
            204, 109,   5, 119, 195,  59, 151,   7, 221, 207],
           dtype='int64', length=258)


,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,Hugo
0,-0.110434,-0.433396,0.460578,-0.016289,-0.028270,-0.541334,0.932202,-1.944028,0.478681,1.281450,...,-3.272984,2.129903,0.047634,-0.508604,0.339272,0.707471,7.873769,1.264210,0.169658,TCGA-BA-4074-01
1,-1.026985,-0.222730,1.373774,0.306599,-1.246431,0.652455,-0.756047,2.044399,-2.785988,0.318557,...,1.579133,-5.189673,1.044104,-0.205798,-8.434000,-3.193501,0.573871,0.055785,1.265552,TCGA-BA-4075-01
2,0.178184,-0.442255,-0.879880,-0.761871,-0.511855,-0.634988,-0.771308,1.731899,-2.374242,-0.860188,...,1.780378,-1.394603,-0.657214,-2.447765,7.683917,-0.117125,0.662364,3.648749,-1.574303,TCGA-BA-4076-01
3,3.268334,-0.952728,2.043752,5.460884,2.188717,3.400235,4.841902,-0.482212,-2.396556,2.968197,...,1.423881,3.028344,-3.673654,-4.089676,-1.310831,0.289284,-2.729687,1.529083,-0.936587,TCGA-BA-4077-01
4,-1.957533,-1.910264,-2.783829,1.897340,-0.524025,0.618983,-0.952788,2.479993,1.022146,1.266803,...,5.747605,7.596730,-2.585597,-0.643833,-5.281845,-5.668230,-0.917955,-4.662671,-1.404119,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,1.679675,3.125836,-3.418426,1.283457,-2.941705,0.006229,0.803040,2.712161,0.183041,-0.906818,...,0.855288,-0.702644,3.888132,-3.157509,-0.326627,-2.981876,-4.001350,-1.844949,-0.525742,TCGA-UF-A7JT-01
515,-2.032022,-1.441785,2.060005,-0.026980,-1.505102,-1.073708,0.090421,0.449611,2.679732,1.800767,...,1.396095,-7.409945,-0.424732,-0.424495,17.389858,3.601498,-3.720558,10.749556,1.606996,TCGA-UF-A7JV-01
516,0.089402,-0.821527,-0.464340,-1.572292,-0.674003,0.568417,-3.456926,0.199807,0.696076,-3.172729,...,-0.022992,-3.434605,-0.986633,-1.109178,-5.310652,-1.872907,5.247226,-5.335525,3.336358,TCGA-UP-A6WW-01
517,3.165689,-2.187939,-7.479713,0.909525,2.314263,0.501657,3.182439,1.488070,1.309657,-0.305723,...,1.145047,-0.201115,0.306857,7.846188,7.764508,-2.094932,14.820934,-4.530487,-7.086287,TCGA-WA-A7GZ-01


In [196]:
df = pd.read_csv("data_linear_CNA.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df5 = df.T
df5.to_csv("CNA.csv")
df5.head()
merged_df = pd.merge(cl, df5, left_index=True, right_index=True, how="inner")
df5=merged_df
c5 = pd.DataFrame(df5)
c5.to_csv("CNApreprocessed.csv")
y = c5['Overall Survival (Months)']
c5 = c5.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
dcn=c5.iloc[:,:]
#print(d)
dcn = dcn.reset_index()
Cn=dcn.iloc[1:,1:]
Cn.head()
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X4 = scaler.fit_transform(Cn[:])
#fit pca on data
pca = TruncatedSVD(n_components=161, algorithm='randomized')
pca.fit(X4)
#transform pca
Z4 =pca.transform(X4)
Z4

n4 = pd.DataFrame(Z4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
n4["Hugo"]=hugo
n4

,0,1,2,3,4,5,6,7,8,9,...,152,153,154,155,156,157,158,159,160,Hugo
0,33.216651,-50.436050,-21.986615,12.885906,11.482549,-24.466906,15.074239,4.380173,5.485482,28.698759,...,-1.665823,0.562289,-7.260205,-0.823600,-2.872974,-1.044169,-4.629931,0.575247,-5.681838,TCGA-BA-4074-01
1,28.236597,64.891526,-91.108694,-44.133534,-34.305138,22.601760,-7.031638,32.943213,-29.234978,6.485621,...,-0.107887,-6.681519,0.188265,2.035563,0.663522,2.878433,-5.464068,-2.250728,-5.229109,TCGA-BA-4075-01
2,31.712084,-19.142462,-44.955697,-10.915900,19.480211,42.130128,-43.298753,-5.824047,-29.996371,34.289886,...,-1.952918,-1.929712,-1.721315,0.281295,-1.136512,-1.328699,0.531922,-0.541963,-0.699381,TCGA-BA-4076-01
3,7.214877,2.055477,16.367214,-31.625528,23.580592,55.697236,-6.511316,-2.483694,-2.420799,28.907656,...,3.502637,-1.491719,-6.297537,8.745909,-0.820567,6.378369,1.126488,-1.338134,3.638809,TCGA-BA-4077-01
4,106.422401,-121.377760,73.987637,-2.817309,61.276928,-18.995732,25.623114,20.360234,25.771941,24.407902,...,-1.729596,2.267675,6.820835,3.202732,-1.992112,1.840634,0.860111,-4.977464,-0.978935,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,-40.190795,-5.455551,3.334912,-12.005291,-13.635163,13.930237,15.611677,-7.288980,-1.165533,-4.699541,...,0.509371,0.843454,0.222985,0.170345,-0.273354,-2.414763,0.480097,0.422347,0.772455,TCGA-UF-A7JT-01
517,-43.964526,-1.144883,-6.521891,-4.225571,-18.626533,-11.038020,7.032690,11.173420,-5.910724,-8.290236,...,2.615444,-1.573423,-0.651542,-1.478751,-1.797448,1.052623,-1.204120,0.192002,-3.121225,TCGA-UF-A7JV-01
518,-11.735949,-119.457581,53.408786,-3.863532,19.905231,-4.851157,-40.494109,-15.368992,-94.071584,20.233563,...,-2.152208,2.262346,-2.595081,-2.924099,0.290634,0.533057,6.320774,-4.521497,2.550472,TCGA-UP-A6WW-01
519,47.663786,-37.292774,-36.082906,-35.697312,-2.236555,-38.248416,-24.300350,23.728644,3.907556,-34.511521,...,-0.834214,-6.498509,-3.086492,5.905371,-3.905229,9.720250,-4.934175,-0.271553,6.085850,TCGA-WA-A7GZ-01


In [197]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z4, y)
n_components = Z4.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(128).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zl4 = np.column_stack(columns)
nl4 = pd.DataFrame(Zl4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nl4["Hugo"]=hugo
nl4

Number of selected features:  128
Selected features:  Int64Index([138, 150, 126, 132, 144, 123, 149, 133, 160, 107,
            ...
             19,   4,  13,  15,  40,   2,   7,   9,  24,  28],
           dtype='int64', length=128)


,0,1,2,3,4,5,6,7,8,9,Hugo
0,0.177764,-0.295297,-6.606839,-0.289890,-0.871307,1.278386,-4.571450,1.677085,-5.681838,2.918783,TCGA-BA-4074-01
1,-4.832561,-6.761386,8.137050,-4.556720,3.022000,-2.715683,-3.597148,5.906409,-5.229109,1.428601,TCGA-BA-4075-01
2,-0.639722,-2.563590,-1.778895,-1.068919,-3.092622,0.205780,0.676152,0.575848,-0.699381,4.402868,TCGA-BA-4076-01
3,11.512247,-3.514193,-12.381864,-16.142227,-7.232444,-6.589102,-2.228914,-8.718956,3.638809,14.454586,TCGA-BA-4077-01
4,5.588925,-0.855556,10.837690,-1.729523,-1.050516,-0.821591,1.469484,0.572703,-0.978935,-2.258615,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...
516,0.460362,-2.694570,0.923820,0.945028,1.857088,-2.372901,-1.088484,-0.402487,0.772455,0.537412,TCGA-UF-A7JT-01
517,0.308286,-0.459329,0.025362,-0.074681,0.192020,1.866719,-0.908094,0.348481,-3.121225,0.411861,TCGA-UF-A7JV-01
518,6.785482,4.858532,1.652761,-0.899516,7.590409,-5.028953,-3.105178,3.112203,2.550472,2.588909,TCGA-UP-A6WW-01
519,-4.327670,-3.191320,6.901149,4.995845,0.162754,11.960026,0.193951,7.485950,6.085850,-2.649948,TCGA-WA-A7GZ-01


In [198]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z4, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z4.shape[1])])
selected_features = coef.abs().nlargest(128).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Ze4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne4 = pd.DataFrame(Ze4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    ne4["Hugo"]=hugo
except Exception:
    pass


Number of selected features:  128
Selected features:  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128]


In [199]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=128)
rfe.fit(Z4, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zre4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre4 = pd.DataFrame(Zre4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    nre4["Hugo"]=hugo
except Exception:
    pass

Selected Features: [2, 4, 7, 8, 9, 12, 13, 15, 17, 18, 19, 21, 24, 25, 28, 29, 30, 32, 33, 36, 37, 39, 40, 44, 45, 46, 47, 49, 50, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 69, 70, 71, 72, 73, 74, 76, 78, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 98, 99, 100, 101, 102, 103, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 128, 129, 130, 131, 132, 133, 134, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161]


In [200]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z4, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z4.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(128)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zrf4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf4 = pd.DataFrame(Zrf4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nrf4["Hugo"]=hugo
nrf4

Number of selected features:  128
Selected features:  [138, 160, 28, 127, 57, 123, 119, 132, 31, 136, 92, 78, 44, 7, 58, 158, 111, 84, 60, 29, 18, 1, 131, 161, 39, 144, 154, 13, 126, 150, 124, 96, 36, 69, 116, 43, 2, 24, 145, 15, 100, 25, 51, 134, 146, 5, 125, 148, 27, 118, 156, 128, 46, 95, 105, 142, 109, 16, 149, 159, 23, 102, 70, 135, 33, 98, 91, 133, 35, 110, 139, 40, 55, 112, 74, 101, 52, 4, 82, 94, 67, 76, 8, 120, 103, 22, 73, 140, 17, 30, 12, 63, 19, 64, 9, 86, 48, 108, 75, 130, 147, 56, 157, 14, 141, 97, 20, 90, 117, 122, 71, 42, 114, 38, 80, 153, 89, 45, 6, 3, 137, 83, 41, 151, 99, 85, 37, 50]


,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,Hugo
0,0.177764,-5.681838,-23.880077,3.429873,-0.275916,1.278386,-0.130403,-0.289890,19.075492,2.423131,...,7.565421,-4.629931,-0.896918,0.130803,-8.354422,-20.368070,0.394163,-50.436050,-1.892662,TCGA-BA-4074-01
1,-4.832561,-5.229109,-45.140033,-15.278478,-8.918660,-2.715683,-2.514814,-4.556720,-3.491042,7.599630,...,-6.466113,-5.464068,-6.778929,2.976174,-16.478756,-35.590843,-11.498770,64.891526,1.828950,TCGA-BA-4075-01
2,-0.639722,-0.699381,4.056920,-6.611510,21.363783,0.205780,-2.359232,-1.068919,2.132905,-1.929689,...,29.368135,0.531922,-4.375048,7.868310,-1.650149,-47.066898,-16.530502,-19.142462,-0.569856,TCGA-BA-4076-01
3,11.512247,3.638809,27.035562,-12.256621,-16.109119,-6.589102,5.638296,-16.142227,19.905749,7.656865,...,5.143998,1.126488,-2.227056,-2.074183,-8.118570,-23.471295,5.027645,2.055477,7.456654,TCGA-BA-4077-01
4,5.588925,-0.978935,-3.693251,-0.989301,-9.510678,-0.821591,6.193741,-1.729523,42.608656,2.483322,...,9.044359,0.860111,-1.450030,-2.596202,-13.525895,4.773220,15.090565,-121.377760,-6.195078,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,0.460362,0.772455,-2.185221,3.248700,2.245474,-2.372901,-0.473348,0.945028,3.486018,0.280729,...,-3.218871,0.480097,2.589060,1.025590,-2.344370,6.022071,9.513237,-5.455551,-0.742631,TCGA-UF-A7JT-01
517,0.308286,-3.121225,-2.834909,-0.201635,2.887759,1.866719,-3.014376,-0.074681,-3.550351,-2.345971,...,-5.767800,-1.204120,1.929553,1.011638,1.479296,4.990147,6.946336,-1.144883,2.842829,TCGA-UF-A7JV-01
518,6.785482,2.550472,-34.445437,-1.978344,10.492119,-5.028953,2.207477,-0.899516,34.614028,-2.919866,...,-8.496452,6.320774,2.157421,3.070354,-4.154714,6.232623,-14.149366,-119.457581,-1.845683,TCGA-UP-A6WW-01
519,-4.327670,6.085850,8.197405,4.637366,17.213745,11.960026,4.812495,4.995845,11.884406,6.187892,...,-1.650501,-4.934175,-1.168948,-9.099187,-10.882739,-15.374603,4.190393,-37.292774,-1.530453,TCGA-WA-A7GZ-01


Merging dataset

In [201]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(n1['Hugo'][i]==n3['Hugo'][j]):
            z.append(n1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==n4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,n1,on='Hugo')
mdf=pd.merge(mdf,n3,on='Hugo')
mdf=pd.merge(mdf,n4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 30
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)




(512, 26)
(512, 872)
[<keras.callbacks.EarlyStopping object at 0x000002591ACB2B50>, <keras.callbacks.ModelCheckpoint object at 0x000002591ACB26A0>]
Epoch 1/30
1/1 [==============================] - 1s 1s/step - loss: 189048.4844
Epoch 2/30
1/1 [==============================] - 0s 79ms/step - loss: 174056.1719
Epoch 3/30
1/1 [==============================] - 0s 79ms/step - loss: 169185.5938
Epoch 4/30
1/1 [==============================] - 0s 78ms/step - loss: 163970.0156
Epoch 5/30
1/1 [==============================] - 0s 31ms/step - loss: 165366.1562
Epoch 6/30
1/1 [==============================] - 0s 79ms/step - loss: 161559.0938
Epoch 7/30
1/1 [==============================] - 0s 94ms/step - loss: 157600.3281
Epoch 8/30
1/1 [==============================] - 0s 16ms/step - loss: 159778.5000
Epoch 9/30
1/1 [==============================] - 0s 63ms/step - loss: 156264.9688
Epoch 10/30
1/1 [==============================] - 0s 78ms/step - loss: 152201.9844
Epoch 11/30
1/1 [======

In [202]:
import deepsurvk

dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)


[<keras.callbacks.EarlyStopping object at 0x0000025895EC2670>, <keras.callbacks.ModelCheckpoint object at 0x0000025894870310>]
Epoch 1/20
1/1 [==============================] - 1s 1s/step - loss: 196084.8125
Epoch 2/20
1/1 [==============================] - 0s 88ms/step - loss: 172256.2031
Epoch 3/20
1/1 [==============================] - 0s 69ms/step - loss: 163954.1719
Epoch 4/20
1/1 [==============================] - 0s 31ms/step - loss: 167183.6875
Epoch 5/20
1/1 [==============================] - 0s 32ms/step - loss: 164547.6562
Epoch 6/20
1/1 [==============================] - 0s 69ms/step - loss: 156968.6562
Epoch 7/20
1/1 [==============================] - 0s 16ms/step - loss: 161856.0156
Epoch 8/20
1/1 [==============================] - 0s 16ms/step - loss: 159911.9375
Epoch 9/20
1/1 [==============================] - 0s 79ms/step - loss: 152723.9844
Epoch 10/20
1/1 [==============================] - 0s 16ms/step - loss: 157438.2812
Epoch 11/20
1/1 [===========================

In [203]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

SVR
C-index:  0.7523987433132376
Mean squared error:  0.007544433406392337
Mean absolute error:  0.05749743108559629
Median absolute error:  0.04400650592353159
R-squared:  0.49281885822182503


In [204]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


XGBOOST
C-index:  0.9899804704084232
Mean squared error:  1.9347983783456506e-05
R-squared:  0.9986993148487354
c_index: 0.9899804704084232
Mean absolute error:  0.0028899391167906473
Median absolute error:  0.0019554670843893035


In [205]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.8950921287254818
Mean squared error:  0.0008808430532825691
R-squared:  0.9407845544620088
c_index: 0.8950921287254818
Mean absolute error:  0.02410319014827482
Median absolute error:  0.023852684873690383


In [206]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

C-index:  0.9953298802751125
Mean squared error:  0.0001421120194331104
R-squared:  0.9904463950579221
c_index: 0.9953298802751125
Mean absolute error:  0.004344668570491869
Median absolute error:  0.001430435607858039


In [207]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nl1['Hugo'][i]==nl3['Hugo'][j]):
            z.append(nl1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nl4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nl1,on='Hugo')
mdf=pd.merge(mdf,nl3,on='Hugo')
mdf=pd.merge(mdf,nl4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 10
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv)")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)



(512, 26)
(512, 176)
[<keras.callbacks.EarlyStopping object at 0x00000259216489A0>, <keras.callbacks.ModelCheckpoint object at 0x0000025921648B50>]
Epoch 1/10
1/1 [==============================] - 1s 841ms/step - loss: 239877.1406
Epoch 2/10
1/1 [==============================] - 0s 48ms/step - loss: 175646.8750
Epoch 3/10
1/1 [==============================] - 0s 0s/step - loss: 232432.3594
Epoch 4/10
1/1 [==============================] - 0s 16ms/step - loss: 200616.7812
Epoch 5/10
1/1 [==============================] - 0s 0s/step - loss: 196037.3594
Epoch 6/10
1/1 [==============================] - 0s 47ms/step - loss: 166817.2812
Epoch 7/10
1/1 [==============================] - 0s 16ms/step - loss: 173093.3594
Epoch 8/10
1/1 [==============================] - 0s 47ms/step - loss: 161184.2969
Epoch 9/10
1/1 [==============================] - 0s 16ms/step - loss: 163556.5312
Epoch 10/10
1/1 [==============================] - 0s 63ms/step - loss: 156889.4062
DeepSurv)
c-index of tra

In [208]:
result = np.zeros((5, 4, 5))

result[0][0][0]=c_index_test
result[0][0][1]=mse
result[0][0][2]=rmse
result[0][0][3]=mae
result[0][0][4]=mdae

In [209]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][0][0]=c_index
result[1][0][1]=mse
result[1][0][2]=r2
result[1][0][3]=mae
result[1][0][4]=mdae

SVR
C-index:  0.9297306483133656
Mean squared error:  0.00035434025517595125
Mean absolute error:  0.014538466839349302
Median absolute error:  0.01209862396081482
R-squared:  0.9783673037337961


In [210]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][0][0]=c_index
result[2][0][1]=mse
result[2][0][2]=r2
result[2][0][3]=mae
result[2][0][4]=mdae

XGBOOST
C-index:  0.9924377602175206
Mean squared error:  5.619122327641027e-05
R-squared:  0.9965694903466121
c_index: 0.9924377602175206
Mean absolute error:  0.0027948591996859386
Median absolute error:  0.0016406080638269316


In [211]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][0][0]=c_index
result[3][0][1]=mse
result[3][0][2]=r2
result[3][0][3]=mae
result[3][0][4]=mdae

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9226357379556462
Mean squared error:  0.0009993294748177446
R-squared:  0.9389903047062417
c_index: 0.9226357379556462
Mean absolute error:  0.025007549843707903
Median absolute error:  0.02478179521152045


In [212]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][0][0]=c_index
result[4][0][1]=mse
result[4][0][2]=r2
result[4][0][3]=mae
result[4][0][4]=mdae

C-index:  0.9977908063556802
Mean squared error:  8.670541471743553e-05
R-squared:  0.9947065796961563
c_index: 0.9977908063556802
Mean absolute error:  0.002942463723995473
Median absolute error:  0.0008019360349245212


In [213]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(ne1['Hugo'][i]==ne3['Hugo'][j]):
            z.append(ne1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==ne4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,ne1,on='Hugo')
mdf=pd.merge(mdf,ne3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][1][0]=c_index_test
result[0][1][1]=mse
result[0][1][2]=rmse
result[0][1][3]=mae
result[0][1][4]=mdae



(512, 26)
(512, 697)
[<keras.callbacks.EarlyStopping object at 0x000002588F892760>, <keras.callbacks.ModelCheckpoint object at 0x00000258921C14C0>]
Epoch 1/20
1/1 [==============================] - 1s 925ms/step - loss: 175717.0469
Epoch 2/20
1/1 [==============================] - 0s 68ms/step - loss: 159357.9219
Epoch 3/20
1/1 [==============================] - 0s 74ms/step - loss: 151672.8906
Epoch 4/20
1/1 [==============================] - 0s 63ms/step - loss: 150959.4531
Epoch 5/20
1/1 [==============================] - 0s 64ms/step - loss: 146966.3750
Epoch 6/20
1/1 [==============================] - 0s 15ms/step - loss: 147154.2031
Epoch 7/20
1/1 [==============================] - 0s 78ms/step - loss: 146308.5156
Epoch 8/20
1/1 [==============================] - 0s 55ms/step - loss: 139262.0156
Epoch 9/20
1/1 [==============================] - 0s 31ms/step - loss: 139731.2656
Epoch 10/20
1/1 [==============================] - 0s 63ms/step - loss: 135438.2344
Epoch 11/20
1/1 [===

In [214]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][1][0]=c_index
result[1][1][1]=mse
result[1][1][2]=r2
result[1][1][3]=mae
result[1][1][4]=mdae

SVR
C-index:  0.7869757174392936
Mean squared error:  0.011510778741947045
Mean absolute error:  0.06751114501013915
Median absolute error:  0.048013648999580154
R-squared:  0.5413659630509249


In [215]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][1][0]=c_index
result[2][1][1]=mse
result[2][1][2]=r2
result[2][1][3]=mae
result[2][1][4]=mdae

XGBOOST
C-index:  0.9913397860417728
Mean squared error:  0.00024819672147380356
R-squared:  0.9901108806902668
c_index: 0.9913397860417728
Mean absolute error:  0.004456576078432361
Median absolute error:  0.0017511256969316123


In [216]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][1][0]=c_index
result[3][1][1]=mse
result[3][1][2]=r2
result[3][1][3]=mae
result[3][1][4]=mdae

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9076243844455765
Mean squared error:  0.0024091327291539326
R-squared:  0.9040108150900731
c_index: 0.9076243844455765
Mean absolute error:  0.030589021322092087
Median absolute error:  0.027409493306959104


In [217]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][1][0]=c_index
result[4][1][1]=mse
result[4][1][2]=r2
result[4][1][3]=mae
result[4][1][4]=mdae

C-index:  0.9963491254881983
Mean squared error:  0.0016331081861790948
R-squared:  0.9349306404898191
c_index: 0.9963491254881983
Mean absolute error:  0.009905718747419434
Median absolute error:  0.0010071652272943751


In [218]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nre1['Hugo'][i]==nre3['Hugo'][j]):
            z.append(nre1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nre4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nre1,on='Hugo')
mdf=pd.merge(mdf,nre3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][2][0]=c_index_test
result[0][2][1]=mse
result[0][2][2]=rmse
result[0][2][3]=mae
result[0][2][4]=mdae



(512, 26)
(512, 695)
[<keras.callbacks.EarlyStopping object at 0x0000025895EC2970>, <keras.callbacks.ModelCheckpoint object at 0x0000025895EC2DF0>]
Epoch 1/20
1/1 [==============================] - 1s 1s/step - loss: 199738.4062
Epoch 2/20
1/1 [==============================] - 0s 79ms/step - loss: 197910.8906
Epoch 3/20
1/1 [==============================] - 0s 79ms/step - loss: 187694.8906
Epoch 4/20
1/1 [==============================] - 0s 78ms/step - loss: 171657.6875
Epoch 5/20
1/1 [==============================] - 0s 16ms/step - loss: 175116.6094
Epoch 6/20
1/1 [==============================] - 0s 16ms/step - loss: 175144.5156
Epoch 7/20
1/1 [==============================] - 0s 72ms/step - loss: 167556.2812
Epoch 8/20
1/1 [==============================] - 0s 53ms/step - loss: 163236.4688
Epoch 9/20
1/1 [==============================] - 0s 16ms/step - loss: 164421.0781
Epoch 10/20
1/1 [==============================] - 0s 78ms/step - loss: 162221.8750
Epoch 11/20
1/1 [======

In [219]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][2][0]=c_index
result[1][2][1]=mse
result[1][2][2]=r2
result[1][2][3]=mae
result[1][2][4]=mdae

SVR
C-index:  0.7597281223449448
Mean squared error:  0.009439178746867777
Mean absolute error:  0.06320481924473906
Median absolute error:  0.04213654590406525
R-squared:  0.545812898725867


In [220]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][2][0]=c_index
result[2][2][1]=mse
result[2][2][2]=r2
result[2][2][3]=mae
result[2][2][4]=mdae

XGBOOST
C-index:  0.9890399320305863
Mean squared error:  8.829229997213127e-05
R-squared:  0.9957516194083648
c_index: 0.9890399320305863
Mean absolute error:  0.0049737053801709305
Median absolute error:  0.0026643791433879523


In [221]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][2][0]=c_index
result[3][2][1]=mse
result[3][2][2]=r2
result[3][2][3]=mae
result[3][2][4]=mdae

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9204757858963466
Mean squared error:  0.0025343335963341393
R-squared:  0.8780548964429125
c_index: 0.9204757858963466
Mean absolute error:  0.028915544231348553
Median absolute error:  0.01894704177695766


In [222]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][2][0]=c_index
result[4][2][1]=mse
result[4][2][2]=r2
result[4][2][3]=mae
result[4][2][4]=mdae

C-index:  0.9941376380628717
Mean squared error:  0.0008961361483558854
R-squared:  0.9568804140186681
c_index: 0.9941376380628717
Mean absolute error:  0.007972845224434854
Median absolute error:  0.001040381512764539


In [223]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nrf1['Hugo'][i]==nrf3['Hugo'][j]):
            z.append(nrf1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nrf4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nrf1,on='Hugo')
mdf=pd.merge(mdf,nrf3,on='Hugo')
mdf=pd.merge(mdf,nrf4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][3][0]=c_index_test
result[0][3][1]=mse
result[0][3][2]=rmse
result[0][3][3]=mae
result[0][3][4]=mdae



(512, 26)
(512, 246)
[<keras.callbacks.EarlyStopping object at 0x0000025895D3B160>, <keras.callbacks.ModelCheckpoint object at 0x0000025921648D00>]
Epoch 1/20
1/1 [==============================] - 1s 865ms/step - loss: 191390.1094
Epoch 2/20
1/1 [==============================] - 0s 47ms/step - loss: 172296.4375
Epoch 3/20
1/1 [==============================] - 0s 62ms/step - loss: 168477.7812
Epoch 4/20
1/1 [==============================] - 0s 48ms/step - loss: 160697.2969
Epoch 5/20
1/1 [==============================] - 0s 47ms/step - loss: 145083.3906
Epoch 6/20
1/1 [==============================] - 0s 16ms/step - loss: 146577.8594
Epoch 7/20
1/1 [==============================] - 0s 47ms/step - loss: 140965.0000
Epoch 8/20
1/1 [==============================] - 0s 47ms/step - loss: 139753.6562
Epoch 9/20
1/1 [==============================] - 0s 45ms/step - loss: 139188.0781
Epoch 10/20
1/1 [==============================] - 0s 31ms/step - loss: 128975.8594
Epoch 11/20
1/1 [===

In [224]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][3][0]=c_index
result[1][3][1]=mse
result[1][3][2]=r2
result[1][3][3]=mae
result[1][3][4]=mdae

SVR
C-index:  0.8553838315217391
Mean squared error:  0.0016376301582741964
Mean absolute error:  0.030483774336892887
Median absolute error:  0.02501995273238329
R-squared:  0.9322865296335452


In [225]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][3][0]=c_index
result[2][3][1]=mse
result[2][3][2]=r2
result[2][3][3]=mae
result[2][3][4]=mdae

XGBOOST
C-index:  0.9936311141304348
Mean squared error:  0.0006065377070681078
R-squared:  0.9749206053355931
c_index: 0.9936311141304348
Mean absolute error:  0.006200402598359022
Median absolute error:  0.001084425065465272


In [226]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][3][0]=c_index
result[3][3][1]=mse
result[3][3][2]=r2
result[3][3][3]=mae
result[3][3][4]=mdae

G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
G:\ana\lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version

C-index:  0.9219174592391305
Mean squared error:  0.0028623358973094543
R-squared:  0.8816468443854516
c_index: 0.9219174592391305
Mean absolute error:  0.028690921049922716
Median absolute error:  0.022385922128018983


In [227]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][3][0]=c_index
result[4][3][1]=mse
result[4][3][2]=r2
result[4][3][3]=mae
result[4][3][4]=mdae

C-index:  0.9954144021739131
Mean squared error:  0.0011269575853632967
R-squared:  0.9534020494950044
c_index: 0.9954144021739131
Mean absolute error:  0.008422528406394791
Median absolute error:  0.000829695359210493


In [228]:
!pip install pandas openpyxl
data_0_0 = result[0, 0:4, 0]
data_1_0 = result[3, 0:4, 0]
data_2_0 = result[2, 0:4, 0]
data_3_0 = result[1, 0:4, 0]
data_4_0 = result[4, 0:4, 0]
combined_data_0 = np.concatenate((data_0_0, data_1_0, data_2_0, data_3_0, data_4_0))
data_0_1 = result[0, 0:4, 1]
data_1_1 = result[3, 0:4, 1]
data_2_1 = result[2, 0:4, 1]
data_3_1 = result[1, 0:4, 1]
data_4_1 = result[4, 0:4, 1]
combined_data_1 = np.concatenate((data_0_1, data_1_1, data_2_1, data_3_1, data_4_1))
data_0_2 = result[0, 0:4, 2]
data_1_2 = result[3, 0:4, 2]
data_2_2 = result[2, 0:4, 2]
data_3_2 = result[1, 0:4, 2]
data_4_2 = result[4, 0:4, 2]
combined_data_2 = np.concatenate((data_0_2, data_1_2, data_2_2, data_3_2, data_4_2))
data_0_3 = result[0, 0:4, 3]
data_1_3 = result[3, 0:4, 3]
data_2_3 = result[2, 0:4, 3]
data_3_3 = result[1, 0:4, 3]
data_4_3 = result[4, 0:4, 3]
combined_data_3 = np.concatenate((data_0_3, data_1_3, data_2_3, data_3_3, data_4_3))
data_0_4 = result[0, 0:4, 4]
data_1_4 = result[3, 0:4, 4]
data_2_4 = result[2, 0:4, 4]
data_3_4 = result[1, 0:4, 4]
data_4_4 = result[4, 0:4, 4]
combined_data_4 = np.concatenate((data_0_4, data_1_4, data_2_4, data_3_4, data_4_4))
# Create a DataFrame
df = pd.DataFrame({
    'Combined_Data_0': combined_data_0,
    'Combined_Data_1': combined_data_1,
    'Combined_Data_2': combined_data_2,
    'Combined_Data_3': combined_data_3,
    'Combined_Data_4': combined_data_4
})

# Save the DataFrame to an Excel file
df.to_excel('combined_data.xlsx', index=False)

print("Data has been saved to combined_data.xlsx")

Data has been saved to combined_data.xlsx
